In [10]:
import geopandas
import lonboard
import pyarrow as pa
import seaborn as sns

filters = [("genus", "==", "Gadus")]
gdf = geopandas.read_parquet("../build/h3_7/data.parquet", filters=filters)[
    ["cell", "records", "geometry", "species"]
]
# DuckDB GeoParquet omits pandas string metadata (columns load as object).
# Lonboard's categorical colormap needs a PyArrow string array, not a pandas Series.
species = pa.array(gdf["species"].astype(str))

In [11]:
unique_species = gdf["species"].astype(str).unique()
# Qualitative palette: evenly spaced hues so species stay visually distinct
palette = sns.color_palette("husl", len(unique_species))
color_map = {
    name: [int(r * 255), int(g * 255), int(b * 255)]
    for name, (r, g, b) in zip(unique_species, palette)
}

point_layer = lonboard.ScatterplotLayer.from_geopandas(gdf)
point_layer.get_radius = 10000
point_layer.radius_max_pixels = 2
point_layer.get_fill_color = lonboard.colormap.apply_categorical_cmap(species, color_map)
lonboard.Map([point_layer])


Map(custom_attribution='', layers=(ScatterplotLayer(get_fill_color=arro3.core.ChunkedArray<FixedSizeList(Field…